# CLI tests — полный каскад извлечения фактов

Тот же пайплайн, что `python -m src.cli.cli`:

```text
query → rules (labeling.py) → CRF NER → fuzzy → brand clf → ATTR typer → JSON
```

Модели: `models/ner_crf.pkl`, `models/brand_clf.joblib`, `models/attr_type_clf.joblib`  
Словари: `artifacts/dicts/`

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cli.cli import DEMO_QUERIES, format_pretty, load_extractor

ex = load_extractor()
print(
    f"crf={ex.ner_model is not None} brand_clf={ex.brand_classifier is not None} "
    f"attr_clf={ex.use_attr_clf} "
    f"dicts B/C/M={len(ex.labeler.brands)}/{len(ex.labeler.categories)}/{len(ex.labeler.models)}"
)

d:\Projects-26-06-2026\mvideo-ner-search\.venv\Lib\site-packages\pymorphy2\analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


crf=True brand_clf=True attr_clf=True dicts B/C/M=1609/1431/6130


d:\Projects-26-06-2026\mvideo-ner-search\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Projects-26-06-2026\mvideo-ner-search\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Projects-26-06-2026\mvideo-ner-search\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression

## Один запрос

In [3]:
q = "ксяоми майнкрафт"
r = ex.extract_debug(q)
print(format_pretty(r, debug=True))
print()
print(json.dumps({k: r[k] for k in ("brand", "category", "model", "attributes", "latency_ms")}, ensure_ascii=False, indent=2))

query:      'ксяоми майнкрафт'
brand:      Xiaomi
category:   ксяоми майнкрафт
model:      None
attributes: {}
latency:    37.156 ms
entities:
  - CATEGORY 'ксяоми майнкрафт'  [0:16]
  - BRAND    'ксяоми'  [0:6]
debug:
  layers: crf=True brand_clf=True attr_clf=True dicts B/C/M=1609/1431/6130
  rules BIO: ксяоми/O майнкрафт/O
  CRF   BIO: ксяоми/B-CATEGORY майнкрафт/I-CATEGORY

{
  "brand": "Xiaomi",
  "category": "ксяоми майнкрафт",
  "model": null,
  "attributes": {},
  "latency_ms": 37.156
}


## Только CRF (без правил / brand clf / ATTR typer)

Каскад **не трогаем** — здесь просто смотрим, что выдаст `ner_crf.pkl` сам по себе: BIO по токенам + spans. Для сравнения рядом — BIO от rules.

In [23]:
from src.ner.labeling import bio_to_entities

assert ex.ner_model is not None, "models/ner_crf.pkl не загружен"


def crf_only(query: str, *, compare_rules: bool = True) -> dict:
    """Разметка только CRF. Каскад extract() не вызывается."""
    crf_pairs = ex.ner_model.predict_query(query)  # [(token, BIO), ...]
    crf_ents = bio_to_entities(crf_pairs, query=query)
    out = {
        "query": query,
        "crf_bio": " ".join(f"{t}/{tag}" for t, tag in crf_pairs),
        "crf_entities": [
            {"text": e["text"], "label": e["label"], "span": e.get("span")} for e in crf_ents
        ],
    }
    if compare_rules:
        rules_pairs = ex.labeler.label_query(query)
        out["rules_bio"] = " ".join(f"{t}/{tag}" for t, tag in rules_pairs)
        out["diff"] = crf_pairs != rules_pairs
    return out


# один запрос — удобно крутить
q_crf = "телефон 16"
r_crf = crf_only(q_crf)
print(f"query:     {r_crf['query']!r}")
print(f"CRF  BIO:  {r_crf['crf_bio']}")
print(f"rules BIO: {r_crf['rules_bio']}")
print(f"diff:      {r_crf['diff']}")
print("CRF entities:")
for e in r_crf["crf_entities"]:
    print(f"  - {e['label']:8} {e['text']!r}  {e.get('span')}")
print()
print(json.dumps(r_crf, ensure_ascii=False, indent=2))

query:     'телефон 16'
CRF  BIO:  телефон/B-CATEGORY 16/O
rules BIO: телефон/B-CATEGORY 16/O
diff:      False
CRF entities:
  - CATEGORY 'телефон'  [0, 7]

{
  "query": "телефон 16",
  "crf_bio": "телефон/B-CATEGORY 16/O",
  "crf_entities": [
    {
      "text": "телефон",
      "label": "CATEGORY",
      "span": [
        0,
        7
      ]
    }
  ],
  "rules_bio": "телефон/B-CATEGORY 16/O",
  "diff": false
}


In [ ]:
# CRF-only по демо-набору (+ свои строки в конце списка)
crf_queries = DEMO_QUERIES + [
    "ксяоми майнкрафт",
    "смартфон xiaomi 256 гб",
]
crf_rows = []
for q in crf_queries:
    r = crf_only(q)
    print(f"query:     {q!r}")
    print(f"CRF  BIO:  {r['crf_bio']}")
    print(f"rules BIO: {r['rules_bio']}")
    print(f"diff={r['diff']}  ents={r['crf_entities']}")
    print("-" * 60)
    crf_rows.append(
        {
            "query": q,
            "crf_bio": r["crf_bio"],
            "rules_bio": r["rules_bio"],
            "diff": r["diff"],
            "crf_labels": ",".join(e["label"] for e in r["crf_entities"]),
        }
    )

import pandas as pd

pd.DataFrame(crf_rows)

## Демо-набор

In [5]:
rows = []
for q in DEMO_QUERIES:
    r = ex.extract(q)
    rows.append(
        {
            "query": q,
            "brand": r.get("brand"),
            "category": r.get("category"),
            "model": r.get("model"),
            "attributes": json.dumps(r.get("attributes") or {}, ensure_ascii=False),
            "n_ents": len(r.get("entities") or []),
            "ms": r.get("latency_ms"),
        }
    )
    print(format_pretty(r))
    print("-" * 60)

import pandas as pd

pd.DataFrame(rows)

query:      'ноутбук asus zenbook 16 гб серый'
brand:      ASUS
category:   ноутбуки
model:      None
attributes: {"memory_storage": "16 гб", "color": "серый"}
latency:    32.282 ms
entities:
  - CATEGORY 'ноутбук'  [0:7]
  - BRAND    'asus'  [8:12]
  - ATTR     '16 гб'  [21:26] type=memory_storage
  - ATTR     'серый'  [27:32] type=color
------------------------------------------------------------
query:      'пылесос dyson v15'
brand:      Dyson
category:   пылесос
model:      v15
attributes: {}
latency:    8.598 ms
entities:
  - CATEGORY 'пылесос'  [0:7]
  - BRAND    'dyson'  [8:13]
  - MODEL    'v15'  [14:17]
------------------------------------------------------------
query:      'айфон 15 pro max'
brand:      Apple
category:   айфон15
model:      15 pro
attributes: {}
latency:    11.081 ms
entities:
  - BRAND    'айфон'  [0:5]
  - MODEL    '15 pro'  [6:12]
  - BRAND    'max'  [13:16]
  - CATEGORY 'айфон 15'  [0:8]
------------------------------------------------------------
query

,query,brand,category,model,attributes,n_ents,ms
0,ноутбук asus zenbook 16 гб серый,ASUS,ноутбуки,NaN,"{""memory_storage"": ""16 гб"", ""color"": ""серый""}",4,32.282
1,пылесос dyson v15,Dyson,пылесос,v15,{},3,8.598
2,айфон 15 pro max,Apple,айфон15,15 pro,{},4,11.081
3,беспроводные наушники sony,Sony,беспроводные наушники,NaN,{},2,8.673
4,телевизор samsung 55 дюйм,Samsung,телевизоры,NaN,"{""size"": ""55 дюйм""}",3,14.916
5,холодильник,NaN,холодильники,NaN,{},1,8.644
6,кофемашина delonghi для капсул,DeLonghi,кофемашина,NaN,"{""UNKNOWN"": ""капсул""}",3,17.643


## Свои запросы

Правь список и перезапусти ячейку.

In [6]:
my_queries = [
    "смартфон xiaomi 256 гб",
    "монитор 27 дюйм 4k",
    "наушники jbl",
]
for q in my_queries:
    print(format_pretty(ex.extract_debug(q), debug=True))
    print("=" * 60)

query:      'смартфон xiaomi 256 гб'
brand:      Xiaomi
category:   смартфон
model:      None
attributes: {"memory_storage": "256 гб"}
latency:    21.09 ms
entities:
  - CATEGORY 'смартфон'  [0:8]
  - BRAND    'xiaomi'  [9:15]
  - ATTR     '256 гб'  [16:22] type=memory_storage
debug:
  layers: crf=True brand_clf=True attr_clf=True dicts B/C/M=1609/1431/6130
  rules BIO: смартфон/B-CATEGORY xiaomi/B-BRAND 256/B-ATTR гб/I-ATTR
  CRF   BIO: смартфон/B-CATEGORY xiaomi/B-BRAND 256/B-ATTR гб/I-ATTR
query:      'монитор 27 дюйм 4k'
brand:      None
category:   мониторы
model:      4 k
attributes: {"size": "27 дюйм"}
latency:    28.021 ms
entities:
  - CATEGORY 'монитор'  [0:7]
  - ATTR     '27 дюйм'  [8:15] type=size
  - MODEL    '4 k'  [16:19]
debug:
  layers: crf=True brand_clf=True attr_clf=True dicts B/C/M=1609/1431/6130
  rules BIO: монитор/B-CATEGORY 27/B-ATTR дюйм/I-ATTR 4/B-MODEL k/I-MODEL
  CRF   BIO: монитор/B-CATEGORY 27/B-ATTR дюйм/I-ATTR 4/B-MODEL k/I-MODEL
query:      'наушники 

## Эквивалент CLI

```bash
python -m src.cli.cli "ноутбук asus 16 гб"
python -m src.cli.cli --demo --debug
python -m src.cli.cli -i
```